# MeshBVH and Self-Intersection
In this section, we create a completely random triangle soup, construct a `TriangleMesh`, and quickly identify all self-intersecting triangles directly using the mesh's built-in BVH methods.

In [1]:
import torch
import plotly.graph_objects as go
import conquer3d

# Create random triangle soup with both intersecting and non-intersecting triangles
N_soup = 20
torch.manual_seed(42)

# Generate random centers in a large space so most are isolated
centers = torch.rand((N_soup, 1, 3), dtype=torch.float32, device="cuda") * 3

# Force some triangles to be near each other to guarantee intersections
centers[1] = centers[0] + torch.rand((1, 1, 3), device="cuda") * 0.5
centers[3] = centers[2] + torch.rand((1, 1, 3), device="cuda") * 0.5
centers[5] = centers[4] + torch.rand((1, 1, 3), device="cuda") * 0.5

# Add small random offsets to create the 3 vertices of each triangle
offsets = torch.rand((N_soup, 3, 3), dtype=torch.float32, device="cuda") * 2
vertices_soup = (centers + offsets).view(N_soup * 3, 3)
triangles_soup = torch.arange(N_soup * 3, dtype=torch.int32, device="cuda").view(N_soup, 3)

# Create mesh
mesh_soup = conquer3d.data_structure.TriangleMesh(vertices_soup, triangles_soup)

# Check self intersection directly from mesh
has_intersection = mesh_soup.is_self_intersection()
intersecting_pairs = mesh_soup.get_self_intersection()

print(f"Has self intersection: {has_intersection}")
print(f"Found {intersecting_pairs.size(0)} intersecting pairs!")

# Visualize
fig = go.Figure()
v0 = vertices_soup[triangles_soup[:, 0]].cpu().numpy()
v1 = vertices_soup[triangles_soup[:, 1]].cpu().numpy()
v2 = vertices_soup[triangles_soup[:, 2]].cpu().numpy()

intersecting_indices = set()
if intersecting_pairs.size(0) > 0:
    intersecting_indices = set(intersecting_pairs.cpu().numpy().flatten())

for i in range(N_soup):
    x = [v0[i, 0], v1[i, 0], v2[i, 0], v0[i, 0]]
    y = [v0[i, 1], v1[i, 1], v2[i, 1], v0[i, 1]]
    z = [v0[i, 2], v1[i, 2], v2[i, 2], v0[i, 2]]
    
    is_intersecting = i in intersecting_indices
    color = 'red' if is_intersecting else 'blue'
    width = 5 if is_intersecting else 2
    name = f"Intersecting Triangle {i}" if is_intersecting else f"Triangle {i}"
    
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color=color, width=width), name=name))

fig.update_layout(scene=dict(aspectmode='data'), title="MeshBVH Self Intersection on Triangle Soup")
fig.show()

Has self intersection: True
Found 5 intersecting pairs!
